In [2]:
import os
import random
import json

In [5]:
root_dir = './experiment_clusters'
num_examples = 100
group_size = 5

In [9]:
clusters = {}
for cluster_name in os.listdir(root_dir):

    cluster_path = os.path.join(root_dir, cluster_name)
    if not os.path.isdir(cluster_path):
        continue
    try:
        first, second = cluster_name.split('_')
        first = int(first) if first else None
        second = int(second)
    except ValueError:

        continue
 
    plans_dir = os.path.join(cluster_path, 'floorplan_reoriented')
    if not os.path.isdir(plans_dir):
        continue

    plan_ids = [os.path.splitext(f)[0] for f in os.listdir(plans_dir) if f.endswith('.png')]
    if len(plan_ids) < 4:
        continue
    clusters[(first, second)] = plan_ids

def sample_group_same_first_global(base_key, all_keys):
    
    # aggregate all plan_ids from clusters sharing the same first label as base_key
    same_first_keys = [k for k in all_keys if k[0] == base_key[0]]
    base_pool = sum((clusters[k] for k in same_first_keys), [])
    if len(base_pool) < group_size:
        return None
    # sample group_size-1 from this pool
    selected = random.sample(base_pool, group_size - 1)

    # now pick an outlier from any cluster whose first label != base_key[0]
    candidates = [k for k in all_keys if k[0] != base_key[0]]
    if not candidates:
        return None
    other_key = random.choice(candidates)
    other_id = random.choice(clusters[other_key])

    return {
        'base_cluster': base_key[0],
        'other_cluster': other_key,
        'group': selected + [other_id],
        'outlier_id': other_id
    }


def sample_group(base_key, all_keys, mode="default"):
    base_ids = clusters[base_key]
    selected = random.sample(base_ids, group_size - 1)

    if mode not in ["default", "same_first", "same_second"]:
        raise Exception('mode not in one of ["default", "same_first", "same_second"]')
    
    if mode == "same_second":
        candidates = [k for k in all_keys if k[1] == base_key[1] and k[0] != base_key[0]]
    elif mode == "same_first":
        candidates = [k for k in all_keys if k[0] == base_key[0] and k[1] != base_key[1]]
    else:
        candidates = [k for k in all_keys if k[1] != base_key[1] and k[0] != base_key[0]]
    if not candidates:
        return None
    other_key = random.choice(candidates)
    other_id = random.choice(clusters[other_key])

    return {
        'base_cluster': base_key,
        'other_cluster': other_key,
        'group': selected + [other_id],
        'outlier_id': other_id
    }


In [ ]:

groups_diff_first = []
groups_same_second = []


while len(groups_diff_first) < num_examples:
    base = random.choice(keys)
    grp = sample_group(base, keys)
    if grp:
        groups_diff_first.append(grp)

while len(groups_same_second) < num_examples:
    base = random.choice(keys)
    grp = sample_group(base, keys, mode="same_second")
    if grp:
        groups_same_second.append(grp)


with open('groups_diff_first.json', 'w') as f:
    json.dump(groups_diff_first, f, indent=2)

with open('groups_same_second.json', 'w') as f:
    json.dump(groups_same_second, f, indent=2)

print(f"Generated {len(groups_diff_first)} 'diff_first' groups and {len(groups_same_second)} 'same_second' groups")




Generated 100 'diff_first' groups and 100 'same_second' groups


In [10]:
groups_same_first_global = []

all_keys = list(clusters.keys())

while len(groups_same_first_global) < num_examples:
    base = random.choice(all_keys)
    grp = sample_group_same_first_global(base, all_keys)
    if grp:
        groups_same_first_global.append(grp)

# write out
with open('groups_same_first_global.json', 'w') as f:
    json.dump(groups_same_first_global, f, indent=2)

print(f"Generated {len(groups_same_first_global)} 'same_first_global' groups")


Generated 100 'same_first_global' groups


In [10]:
groups_same_first = []
keys = list(clusters.keys())

while len(groups_same_first) < num_examples:
    base = random.choice(keys)
    grp = sample_group(base, keys, mode="same_first")
    if grp:
        groups_same_first.append(grp)

with open('groups_same_first.json', 'w') as f:
    json.dump(groups_same_first, f, indent=2)

print(f"Generated {len(groups_same_first)} 'same_first' groups")

Generated 100 'same_first' groups


In [ ]:

root_dir = 'experiment_clusters' 
num_examples = 100 
group_size = 5
templates_dir = 'prompts'  
output_dir = './'  


with open(os.path.join(templates_dir, 'description_pick_diff_prompt.txt')) as f:
    desc_template = f.read()
with open(os.path.join(templates_dir, 'json_pick_diff_prompt.txt')) as f:
    json_template = f.read()

def build_prompt(sample, input_type='description'):
    # ordered retrieval: first group_size-1 from base, then the outlier
    base_ids = sample['group'][:-1]
    outlier = sample['outlier_id']
    items = base_ids + [outlier]
    # shuffle for prompt variability
    random.shuffle(items)

    bullets = []
    for idx, plan_id in enumerate(items, start=1):
    
        if plan_id == outlier:
            cluster = sample['other_cluster']
        else:
            cluster = sample['base_cluster']
        subfolder = 'human_annotation' if input_type == 'description' else 'json'
        ext = 'txt' if input_type == 'description' else 'json'
        if isinstance(cluster, int):
            file_dir = f"{cluster}_"
        else:
            file_dir = f"{cluster[0]}_{cluster[1]}"
        file_path = os.path.join(root_dir, file_dir, subfolder, f"{plan_id}.{ext}")

        if input_type == 'description':
            with open(file_path) as f:
                content = f.read().strip()
            entry = f"Example {idx}:\nID {plan_id}\n{content}\n"
        else:
            with open(file_path) as f:
                data = json.load(f)
            pretty = json.dumps(data, indent=2)
            entry = f"Example {idx}:\nID {plan_id}\n{pretty}\n"
        bullets.append(entry)

    prompt_text = "".join(bullets)
    if input_type == 'description':
        return desc_template.format(num_shots=group_size, num_shots_minus=group_size-1, description=prompt_text)
    else:
        return json_template.format(num_shots=group_size, num_shots_minus=group_size-1, example_prettified_json=prompt_text)

def load_json(fname):
    with open(fname) as f:
        return json.load(f)

# diff_first_samples = load_json('groups_diff_first.json')
# same_second_samples = load_json('groups_same_second.json')
# same_first_samples = load_json('groups_same_first.json')
same_first_samples = load_json('groups_same_first.json')

eval_data = {
    'description': {'same_first': []},
    'json': {'same_first': []}
}

def generate_evals(samples, scenario):
    for sample in samples:
        for input_type in ['description', 'json']:
            prompt = build_prompt(sample, input_type=input_type)
            eval_data[input_type][scenario].append({
                'outlier_id': sample['outlier_id'],
                'prompt': prompt
            })

# generate_evals(diff_first_samples, 'diff_first')
# generate_evals(same_second_samples, 'same_second')

# generate_evals(same_first_samples, 'same_first')

# os.makedirs(output_dir, exist_ok=True)
# with open(os.path.join(output_dir, 'eval_prompts_same_first.json'), 'w') as f:
#     json.dump(eval_data, f, indent=2)

# print(f"Built prompts for description and JSON inputs, across both scenarios.")

Built prompts for description and JSON inputs, across both scenarios.


In [18]:

root_dir = '../../' 
num_examples = 100 
group_size = 5
templates_dir = 'prompts'  
output_dir = './'  


with open(os.path.join(templates_dir, 'description_pick_diff_prompt.txt')) as f:
    desc_template = f.read()
with open(os.path.join(templates_dir, 'json_pick_diff_prompt.txt')) as f:
    json_template = f.read()

def build_prompt_general(sample, input_type='description'):
    # ordered retrieval: first group_size-1 from base, then the outlier
    base_ids = sample['group'][:-1]
    outlier = sample['outlier_id']
    items = base_ids + [outlier]
    # shuffle for prompt variability
    random.shuffle(items)

    bullets = []
    for idx, plan_id in enumerate(items, start=1):
    
        if plan_id == outlier:
            cluster = sample['other_cluster']
        else:
            cluster = sample['base_cluster']
        subfolder = 'annotation/human_annotated_tags'
        ext = 'txt'

        file_path = os.path.join(root_dir, subfolder, f"{plan_id}.{ext}")

        
        with open(file_path) as f:
            content = f.read().strip()
        entry = f"Example {idx}:\nID {plan_id}\n{content}\n"

        bullets.append(entry)

    prompt_text = "".join(bullets)

    return desc_template.format(num_shots=group_size, num_shots_minus=group_size-1, description=prompt_text)


def generate_evals_general(samples, scenario):
    for sample in samples:
        for input_type in ['description', 'json']:
            prompt = build_prompt_general(sample, input_type=input_type)
            eval_data[input_type][scenario].append({
                'outlier_id': sample['outlier_id'],
                'prompt': prompt
            })

In [19]:
same_first_global_samples = load_json('groups_same_first_global.json')

# inject a new section into your eval_data dict:
eval_data['description']['same_first_global'] = []
eval_data['json']['same_first_global'] = []

# and then:
generate_evals_general(same_first_global_samples, 'same_first_global')

with open(os.path.join(output_dir, 'eval_prompts_same_first_global.json'), 'w') as f:
    json.dump(eval_data, f, indent=2)

print("Built prompts for the global same‑first scenario.")

Built prompts for the global same‑first scenario.
